# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [ ]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(18):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-270m-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

print(transcoder_paths)  # verify all 18 are present

{0: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_0_width_16k_l0_small/params.safetensors', 1: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_1_width_16k_l0_small/params.safetensors', 2: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_2_width_16k_l0_small/params.safetensors', 3: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_3_width_16k_l0_small/params.safetensors', 4: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_a

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-270m-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-270m",
    transcoders=transcoder_set,       # ← our manually loaded TranscoderSet
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

Loaded pretrained model google/gemma-3-270m into HookedTransformer


In [29]:
# ══════════════════════════════════════════════════════════════════════════════
# CIRCUIT TRACING: Planning vs Execution Stage Analysis
# Mirrors the poetry experiment from the Anthropic attribution-graphs paper.
# Paste this as new cells after your existing "intervention" section.
# ══════════════════════════════════════════════════════════════════════════════

import json
import glob
import re
import requests
from collections import defaultdict
from pathlib import Path


# ── CONFIG ────────────────────────────────────────────────────────────────────
GRAPH_DIR = "./graphs/gemma-3-270m"          # same as your existing cells
RHYME_TOKEN = "it"                            # the token you're rhyming on
RHYME_STEP = 8                                # step index that produces "it"
TOP_N = 15                                    # features to track per step
MODEL_ID = "gemma-scope-2-270m-pt"           # for Neuronpedia lookups
FEAT_WIDTH = "16k"


# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Per-step feature extraction
# Builds a timeline: {step -> [(layer, feat, influence, token_str), ...]}
# ══════════════════════════════════════════════════════════════════════════════

def parse_node_ids(node):
    """Return (layer, local_feat) from node dict, handling both id formats."""
    # Try jsNodeId first (format: "LAYER_FEAT-TOKEN_POS")
    js = node.get("jsNodeId", "")
    m = re.match(r"^(\d+)_(\d+)-", js)
    if m:
        return int(m.group(1)), int(m.group(2))
    # Fallback: use node["layer"] and node["feature"] directly
    layer = node.get("layer")
    feat  = node.get("feature")
    if layer is not None and feat is not None:
        return int(layer), int(feat)
    return None, None


step_timelines = {}   # step_idx -> list of dicts

for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    fname = Path(fpath).stem                  # e.g. "step-08-it"
    m = re.match(r"step-(\d+)-(.+)", fname)
    if not m:
        continue
    step_idx  = int(m.group(1))
    token_str = m.group(2).replace("_", " ")

    with open(fpath) as f:
        data = json.load(f)

    rows = []
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        inf = node.get("influence") or 0
        if inf == 0:
            continue
        layer, feat = parse_node_ids(node)
        if layer is None:
            continue
        rows.append({
            "step":      step_idx,
            "token":     token_str,
            "layer":     layer,
            "feat":      feat,
            "influence": abs(inf),
            "raw_inf":   inf,
        })

    rows.sort(key=lambda x: -x["influence"])
    step_timelines[step_idx] = rows[:TOP_N]

print(f"Loaded {len(step_timelines)} steps: {sorted(step_timelines)}")
for s, rows in sorted(step_timelines.items()):
    tok = rows[0]["token"] if rows else "?"
    print(f"  step {s:02d} '{tok}': {len(rows)} top features")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Feature timeline: track each feature across all steps
# Produces feature_timeline[feat_key] = {step: influence}
# ══════════════════════════════════════════════════════════════════════════════

feature_timeline = defaultdict(lambda: defaultdict(float))
all_steps_tokens = {}   # step -> token_str

for step_idx, rows in step_timelines.items():
    if rows:
        all_steps_tokens[step_idx] = rows[0]["token"]
    for r in rows:
        key = (r["layer"], r["feat"])
        feature_timeline[key][step_idx] = r["influence"]

# Identify features that appear in the rhyme step
rhyme_step_features = set()
if RHYME_STEP in step_timelines:
    for r in step_timelines[RHYME_STEP]:
        rhyme_step_features.add((r["layer"], r["feat"]))

print(f"\nUnique features across all steps: {len(feature_timeline)}")
print(f"Features active at rhyme step {RHYME_STEP}: {len(rhyme_step_features)}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Early-spike detection (planning features)
# A feature is a "planning" feature if its peak influence is BEFORE the
# rhyme step (mirrors the paper's "rhyme-feature activates early" finding).
# ══════════════════════════════════════════════════════════════════════════════

all_step_indices = sorted(step_timelines.keys())

planning_features  = []   # peak before rhyme step
execution_features = []   # peak at or after rhyme step

for feat_key, step_inf in feature_timeline.items():
    if not step_inf:
        continue
    peak_step = max(step_inf, key=step_inf.get)
    peak_val  = step_inf[peak_step]
    rhyme_val = step_inf.get(RHYME_STEP, 0.0)
    span      = sorted(step_inf.keys())

    entry = {
        "feat_key":   feat_key,
        "peak_step":  peak_step,
        "peak_val":   peak_val,
        "rhyme_val":  rhyme_val,
        "first_step": span[0],
        "last_step":  span[-1],
        "n_steps":    len(span),
    }

    if peak_step < RHYME_STEP:
        planning_features.append(entry)
    else:
        execution_features.append(entry)

planning_features.sort(key=lambda x: -x["peak_val"])
execution_features.sort(key=lambda x: -x["peak_val"])

print("=" * 70)
print(f"PLANNING FEATURES  (peak before step {RHYME_STEP}): {len(planning_features)}")
print("=" * 70)
for e in planning_features[:10]:
    l, f = e["feat_key"]
    print(f"  L{l:2d} F{f:5d}  peak_step={e['peak_step']}  peak_inf={e['peak_val']:.4f}  "
          f"rhyme_inf={e['rhyme_val']:.4f}  active_steps={e['n_steps']}")

print()
print("=" * 70)
print(f"EXECUTION FEATURES (peak at step {RHYME_STEP} or after): {len(execution_features)}")
print("=" * 70)
for e in execution_features[:10]:
    l, f = e["feat_key"]
    print(f"  L{l:2d} F{f:5d}  peak_step={e['peak_step']}  peak_inf={e['peak_val']:.4f}  "
          f"rhyme_inf={e['rhyme_val']:.4f}  active_steps={e['n_steps']}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Rhyme-circuit candidate features
# Features that: (a) appear in the rhyme step AND (b) were already active
# at least 2 steps earlier → strongest planning signal
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("RHYME-CIRCUIT CANDIDATES  (active at rhyme AND spiked earlier)")
print("=" * 70)

candidates = []
for e in planning_features:
    if e["feat_key"] in rhyme_step_features:
        candidates.append(e)

candidates.sort(key=lambda x: -(x["peak_val"] + x["rhyme_val"]))

for e in candidates[:15]:
    l, f = e["feat_key"]
    steps_before = RHYME_STEP - e["first_step"]
    print(f"  L{l:2d} F{f:5d}  first_active=step{e['first_step']}  "
          f"({steps_before} steps before rhyme)  "
          f"peak={e['peak_val']:.4f} @ step{e['peak_step']}  "
          f"rhyme_inf={e['rhyme_val']:.4f}")

print(f"\nTotal rhyme-circuit candidates: {len(candidates)}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Per-step top features printed as a table
# Lets you see which layer/features dominate at each generation step
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("PER-STEP TOP FEATURES (step × layer × feature × influence)")
print("=" * 70)

for step_idx in sorted(step_timelines.keys()):
    rows = step_timelines[step_idx]
    tok  = rows[0]["token"] if rows else "?"
    marker = "  ← RHYME" if step_idx == RHYME_STEP else ""
    print(f"\nstep {step_idx:02d} '{tok}'{marker}")
    for r in rows[:5]:
        print(f"    L{r['layer']:2d} F{r['feat']:5d}  inf={r['influence']:.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Neuronpedia feature descriptions
# Fetches labels for top candidates; works for described layers only.
# Set FETCH_DESCRIPTIONS = False to skip if offline / no API access.
# ══════════════════════════════════════════════════════════════════════════════

FETCH_DESCRIPTIONS = False
NP_BASE = "https://www.neuronpedia.org/api/feature"

def fetch_description(layer, feat, model_id=MODEL_ID, width=FEAT_WIDTH):
    """Return Neuronpedia description string or None."""
    # Neuronpedia SAE ID for gemma-scope transcoders
    sae_id = f"{layer}-gemmascope-2-transcoder-{width}"
    url = f"{NP_BASE}/{model_id}/{sae_id}/{feat}"
    try:
        r = requests.get(url, timeout=8)
        if r.status_code != 200:
            return None
        d = r.json()
        # description lives under explanations[0].description
        exps = d.get("explanations") or []
        if exps:
            return exps[0].get("description")
        return d.get("label") or d.get("description")
    except Exception:
        return None


if FETCH_DESCRIPTIONS:
    print("=" * 70)
    print("NEURONPEDIA DESCRIPTIONS FOR TOP RHYME-CIRCUIT CANDIDATES")
    print("=" * 70)

    # Query top candidates + top execution features at rhyme step
    query_set = [(e["feat_key"], "planning")  for e in candidates[:8]] + \
                [(e["feat_key"], "execution") for e in execution_features[:5]
                 if e["feat_key"] in rhyme_step_features]

    for (layer, feat), stage in query_set:
        desc = fetch_description(layer, feat)
        label = desc if desc else "(no description)"
        print(f"  [{stage:9s}] L{layer:2d} F{feat:5d}  →  {label}")
else:
    print("(Neuronpedia fetch disabled — set FETCH_DESCRIPTIONS = True to enable)")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Temporal clustering summary
# Groups features into EARLY (planning), MID, LATE (execution) bands
# and shows which layers dominate each band.
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter

n_steps = len(all_step_indices)
early_cutoff = RHYME_STEP // 3            # first third
mid_cutoff   = (2 * RHYME_STEP) // 3     # second third

bands = {"EARLY (planning)": [], "MID (buildup)": [], "LATE (execution)": []}

for feat_key, step_inf in feature_timeline.items():
    if not step_inf:
        continue
    peak = max(step_inf, key=step_inf.get)
    if peak <= early_cutoff:
        bands["EARLY (planning)"].append(feat_key)
    elif peak <= mid_cutoff:
        bands["MID (buildup)"].append(feat_key)
    else:
        bands["LATE (execution)"].append(feat_key)

print("=" * 70)
print("TEMPORAL CLUSTERING SUMMARY")
print(f"  early ≤ step {early_cutoff}  |  mid ≤ step {mid_cutoff}  |  late > step {mid_cutoff}")
print("=" * 70)

for band_name, feats in bands.items():
    layer_counts = Counter(layer for layer, _ in feats)
    top_layers = layer_counts.most_common(5)
    print(f"\n{band_name}:  {len(feats)} features")
    print(f"  Top layers: " + "  ".join(f"L{l}×{c}" for l, c in top_layers))
    # show the highest-influence feature per band
    top_in_band = sorted(
        feats,
        key=lambda lf: max(feature_timeline[lf].values()),
        reverse=True
    )[:3]
    for lf in top_in_band:
        peak_inf = max(feature_timeline[lf].values())
        peak_s   = max(feature_timeline[lf], key=feature_timeline[lf].get)
        print(f"    L{lf[0]:2d} F{lf[1]:5d}  peak_inf={peak_inf:.4f} @ step{peak_s}")

Loaded 19 steps: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  step 00 'He': 15 top features
  step 01 'saw': 15 top features
  step 02 'a': 15 top features
  step 03 'carrot': 15 top features
  step 04 'and': 15 top features
  step 05 'had': 15 top features
  step 06 'to': 15 top features
  step 07 'grab': 15 top features
  step 08 'it': 15 top features
  step 09 ',': 15 top features
  step 11 'He': 15 top features
  step 12 'saw': 15 top features
  step 13 'a': 15 top features
  step 14 'carrot': 15 top features
  step 15 'and': 15 top features
  step 16 'had': 15 top features
  step 17 'to': 15 top features
  step 18 'grab': 15 top features
  step 19 'it': 15 top features

Unique features across all steps: 263
Features active at rhyme step 8: 15
PLANNING FEATURES  (peak before step 8): 110
  L16 F 4457  peak_step=1  peak_inf=0.8005  rhyme_inf=0.0000  active_steps=1
  L17 F15738  peak_step=3  peak_inf=0.8004  rhyme_inf=0.0000  active_steps=1
  L16 F11845  peak_

In [32]:
# Check if planning features exist BEFORE the rhyme step
rhyme_step_features = {}
if 8 in step_timelines:
    for r in step_timelines[8]:
        key = (r["layer"], r["feat"])
        rhyme_step_features[key] = r["influence"]

print("=" * 70)
print("DID THESE FEATURES SPIKE EARLY? (planning signal)")
print("=" * 70)

early_spikes = []
for feat_key, rhyme_inf in rhyme_step_features.items():
    if feat_key not in feature_timeline:
        continue
    step_inf = feature_timeline[feat_key]
    
    # Was this feature active BEFORE step 8?
    pre_rhyme_steps = {s: v for s, v in step_inf.items() if s < 8}
    if pre_rhyme_steps:
        pre_peak = max(pre_rhyme_steps.values())
        pre_step = max(pre_rhyme_steps, key=pre_rhyme_steps.get)
        early_spikes.append({
            "feat": feat_key,
            "pre_peak": pre_peak,
            "pre_step": pre_step,
            "rhyme_inf": rhyme_inf,
            "ratio": pre_peak / rhyme_inf if rhyme_inf > 0 else float('inf'),
        })

early_spikes.sort(key=lambda x: -x["pre_peak"])

for e in early_spikes[:10]:
    l, f = e["feat"]
    print(f"L{l:2d} F{f:5d}  peaked @ step{e['pre_step']} (inf={e['pre_peak']:.4f})  "
          f"→  rhyme_step_inf={e['rhyme_inf']:.4f}  ratio={e['ratio']:.2f}x")

print(f"\nTotal features that spiked before rhyme: {len(early_spikes)} / {len(rhyme_step_features)}")

DID THESE FEATURES SPIKE EARLY? (planning signal)
L17 F11499  peaked @ step7 (inf=0.7968)  →  rhyme_step_inf=0.7993  ratio=1.00x

Total features that spiked before rhyme: 1 / 15
